In [1]:
import pickle
import re
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.runnables import RunnableLambda
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_149865/1917416524.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
<python-environment>/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuration

In [2]:
index_path = Path("../bm25/bm25_index.pkl")
db_dir = Path("../chroma_context_db")
collection_name = "trec_2019_trials_context"
model_name = "BAAI/bge-small-en-v1.5"
device = "cuda"
candidate_count = 100

### Load BM25

In [3]:
with index_path.open("rb") as file:
    saved_index = pickle.load(file)

documents = []
for metadata in saved_index["metadatas"]:
    document = TextLoader(metadata["source"], encoding="utf-8").load()[0]
    document.metadata.update(metadata)
    documents.append(document)

bm25_retriever = BM25Retriever(
    vectorizer=saved_index["vectorizer"],
    docs=documents,
    preprocess_func=lambda text: re.findall(
        saved_index["token_pattern"],
        text.lower() if saved_index["lowercase"] else text,
    ),
    k=candidate_count,
)
print("BM25 documents:", len(documents))

BM25 documents: 8567


### Load Chroma

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device, "local_files_only": True},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)
vectorstore = Chroma(
    collection_name=collection_name,
    persist_directory=str(db_dir),
    embedding_function=embedding_model,
    create_collection_if_not_exists=False,
)
vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": candidate_count}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1022.26it/s]


### Deduplicate vector results

The vector retriever returns chunks. Keep the first hit per NCT ID before combining trial rankings.

In [5]:
def unique_trials(hits):
    trials = []
    seen = set()
    for document in hits:
        nct_id = document.metadata["nct_id"]
        if nct_id not in seen:
            seen.add(nct_id)
            trials.append(document)
    return trials

vector_trial_retriever = vector_retriever | RunnableLambda(unique_trials)

### Combine rankings with RRF

The vector branch retrieves 100 chunks before deduplication; the BM25 branch retrieves 100 trials. Each unique trial contributes once per branch.

In [6]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_trial_retriever, bm25_retriever],
    weights=[0.5, 0.5],
    c=60,
    id_key="nct_id",
)

### Retrieve candidate trials

In [7]:
query = "melanoma BRAF V600E"
results = hybrid_retriever.invoke(query)[:10]

for rank, document in enumerate(results, start=1):
    print(rank, document.metadata["nct_id"])

1 NCT01264380
2 NCT01543113
3 NCT03101254
4 NCT01586403
5 NCT01248936
6 NCT02202200
7 NCT03455764
8 NCT01942993
9 NCT02120222
10 NCT01037127
